In [4]:
import pandas as pd
import duckdb

In [5]:
database_path = "../olist-ecommerce.duckdb"

In [12]:
#create connection to .duckdb database
con = duckdb.connect(database_path)

In [20]:
#turn each table into a pandas dataframe
df_customers = con.sql( "SELECT * FROM olist_customers").df()
df_order_reviews = con.sql( "SELECT * FROM olist_order_reviews").df()
df_orders = con.sql( "SELECT * FROM olist_orders").df()
df_order_items = con.sql( "SELECT * FROM olist_order_items").df()
df_order_payments = con.sql( "SELECT * FROM olist_order_payments").df()
df_geolocation = con.sql( "SELECT * FROM olist_geolocation").df()
df_sellers = con.sql( "SELECT * FROM olist_sellers").df()
df_products = con.sql( "SELECT * FROM olist_products").df()

In [ ]:
df_orders.

In [21]:
#example eda customers tables
df_customers.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [26]:
#check null values per column
df_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [50]:
df_customers.dtypes

customer_id                 str
customer_unique_id          str
customer_zip_code_prefix    str
customer_city               str
customer_state              str
customer_city_clean         str
customer_city_state         str
dtype: object

In [27]:
#check if states are all written the same way
df_customers['customer_state'].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [29]:
from unidecode import unidecode

In [31]:
df_customers['customer_city_clean'] = (
    
    df_customers['customer_city']
    .str.lower() #lower-case column
    .str.strip() #remove trailling white space
    .apply(unidecode) #remove accents so "são" and "sao" become "sao"
)

In [34]:
#separate potential cities with same name
df_customers['customer_city_state'] = df_customers['customer_city_clean'] + ', ' + df_customers['customer_state']
df_customers.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_city_clean,customer_city_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,franca,"franca, SP"
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,sao bernardo do campo,"sao bernardo do campo, SP"
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,sao paulo,"sao paulo, SP"
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,mogi das cruzes,"mogi das cruzes, SP"
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,campinas,"campinas, SP"


In [51]:
df_customers_mask = (
                df_customers['customer_zip_code_prefix']
                .astype(str)
                .str.strip()
                .str.match(r'^\d{5}$')
)
df_customers_mask.all() #validate whether all customer_zipcode_prefixes are 5 digits

np.True_

In [53]:
df_customers.nunique()

customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
customer_city_clean          4119
customer_city_state          4310
dtype: int64

In [61]:
#not sure what customer_unique_id is, given there are duplicates and there's also a customer_id
df_customers[df_customers['customer_unique_id'].duplicated() == True]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_city_clean,customer_city_state
679,c57b4b6f3719475543b721e720a526ad,b6c083700ca8c135ba9f0f132930d4e8,08686,suzano,SP,suzano,"suzano, SP"
1463,9f6f3da49e2d46e3a7529f5e3c25ecce,a40096fc0a3862e9e12bc55b5f8e6ab2,20561,rio de janeiro,RJ,rio de janeiro,"rio de janeiro, RJ"
1607,299f7b5125c8fbe1761a1b320c34fc7d,b8b3c435a58aebd788a477bed8342910,95585,arroio do sal,RS,arroio do sal,"arroio do sal, RS"
2811,226d59f9f4b98e228b689eea45d03a6d,66980c3775537536f77b434d74e520f5,30290,belo horizonte,MG,belo horizonte,"belo horizonte, MG"
3198,b1253701171dfb298f52a221f824e45b,788313d21c3507fe890921f6e17aa679,11070,santos,SP,santos,"santos, SP"
...,...,...,...,...,...,...,...
99324,5b46a0d983eec8c97363bea78d4a69dd,8bab3162259edfaadd1ea2e1fe7f58dc,31565,belo horizonte,MG,belo horizonte,"belo horizonte, MG"
99327,c1affa46f9f3b514555259049a0307b9,12ab9334b1240d6d037f2b0102a49571,38050,uberaba,MG,uberaba,"uberaba, MG"
99336,ebf46ff530343a129926adc1f831dea4,0ee57f62666561b72f2ceacad0230cbf,09530,sao caetano do sul,SP,sao caetano do sul,"sao caetano do sul, SP"
99353,282fbce48e4d2077aad602dd125c9225,0ceb502fc33a2ad327b08288c5310e2e,29134,viana,ES,viana,"viana, ES"


In [62]:
#checking 1 example to try to figure it out
df_customers[df_customers['customer_unique_id'] == 'a40096fc0a3862e9e12bc55b5f8e6ab2']

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_city_clean,customer_city_state
197,eee74b2a9aa0989243d2793c1a90fbf6,a40096fc0a3862e9e12bc55b5f8e6ab2,20561,rio de janeiro,RJ,rio de janeiro,"rio de janeiro, RJ"
1463,9f6f3da49e2d46e3a7529f5e3c25ecce,a40096fc0a3862e9e12bc55b5f8e6ab2,20561,rio de janeiro,RJ,rio de janeiro,"rio de janeiro, RJ"
35912,039bbed64cc5e2160b7ca4ac6b09449c,a40096fc0a3862e9e12bc55b5f8e6ab2,20561,rio de janeiro,RJ,rio de janeiro,"rio de janeiro, RJ"


In [65]:
#it appears to be the zip_code id, not the customer, let's rename the column
df_customers.rename(columns={"customer_unique_id": "zip_code_unique_id"}, inplace=True)
df_customers.columns.to_list()

['customer_id',
 'zip_code_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'customer_city_clean',
 'customer_city_state']